In [0]:
import requests
import zipfile
import io
import os

base_output_dir = "/Volumes/workspace/default/data/extracted"
os.makedirs(base_output_dir, exist_ok=True)

for year in range(2015, 2026):
    # Changement de l'URL pour viser les fichiers annuels par industrie si disponible
    url = f"https://data.bls.gov/cew/data/files/{year}/csv/{year}_annual_singlefile.zip"
    print(f"--- Tentative pour l'année {year} ---")

    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        
        with zipfile.ZipFile(io.BytesIO(r.content)) as zip_ref:
            extracted_count = 0
            for file_name in zip_ref.namelist():
                # On vérifie si c'est un CSV et s'il contient les mots clés
                if ".annual" in file_name and file_name.endswith(".csv"):
                    print(f"Fichier trouvé : {file_name}")
                    # On nettoie le nom pour éviter de recréer des sous-dossiers inutiles
                    simple_name = os.path.basename(file_name)
                    out_path = os.path.join(base_output_dir, simple_name)
                    
                    with zip_ref.open(file_name) as source, open(out_path, "wb") as target:
                        target.write(source.read())
                    extracted_count += 1
            
            print(f"Succès : {extracted_count} fichiers extraits pour {year}.")

    except Exception as e:
        print(f"Erreur pour {year} : Le fichier n'existe peut-être pas encore ou l'URL a changé.")

print(f"\nTerminé ! Les fichiers sont ici : {base_output_dir}")

In [0]:
# Utilisation de '*' pour lire tous les fichiers CSV du dossier
path = "/Volumes/workspace/default/data/extracted_v/*.csv"

# Lecture globale
df_all = spark.read.csv(path, header=True, inferSchema=True)

# Affichage des informations
print(f"Nombre total de lignes : {df_all.count()}")
print(f"Nombre de colonnes : {len(df_all.columns)}")
print("-" * 30)
print("Liste des colonnes :")
print(df_all.columns)

if "year" in df_all.columns:
    df_all.groupBy("year").count().orderBy("year").show()